## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting?
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Sid Luthra
**Date:** 10/2/26

Q1.1: Regression predicts a continuous numerical value, while classification predicts a category or class label.

Q1.2: A confusion table compares predicted classes to actual classes. It shows how many predictions were correct or incorrect for each class.

Q1.3: SSE, or Sum of Squared Errors, measures the total squared difference between predicted and actual values. A lower SSE generally means the model fits the data better.

Q1.4: Overfitting happens when a model learns the training data too closely and performs poorly on new data. Underfitting happens when a model is too simple to capture important patterns.

Q1.5: Testing different k values on separate data helps determine which value generalizes best to unseen data. This helps reduce overfitting or underfitting.

Q1.6: A class label is simple and easy to interpret but does not show how confident the model is. A probability distribution gives more information about uncertainty, but it is slightly more complex to interpret.

In [18]:
#Q2.1
import pandas as pd

df = pd.read_csv("land_mines.csv")
df.head()

,voltage,height,soil,mine_type
0,0.338157,0.000000,0.0,1
1,0.320241,0.181818,0.0,1
2,0.287009,0.272727,0.0,1
3,0.256284,0.454545,0.0,1
4,0.262840,0.545455,0.0,1


### Q2.1 Load the data and perform EDA
The dataset contains three predictor variables, `voltage`, `height`, and `soil`, and the target variable `mine_type`, which contains classes 1 through 5.

In [19]:
print(df.shape)
print(df.info())
print(df.describe())
print(df["mine_type"].value_counts().sort_index())

(338, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
None
          voltage      height        soil   mine_type
count  338.000000  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550    2.952663
std      0.195819    0.306043    0.344244    1.419703
min      0.197734    0.000000    0.000000    1.000000
25%      0.309737    0.272727    0.200000    2.000000
50%      0.359516    0.545455    0.600000    3.000000
75%      0.482628    0.727273    0.800000    4.000000
max      0.999999    1.000000    1.000000    5.000000
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64


In [20]:
from sklearn.model_selection import train_test_split

X = df[["voltage", "height", "soil"]]
y = df["mine_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.50,
    random_state=42,
    stratify=y
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

Training observations: 169
Testing observations: 169


### Q2.2 Train/test split
I split the data 50/50 into training and testing sets. I used stratification so that each mine type is represented proportionally in both sets.

In [21]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

k_values = range(1, 31)
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    accuracies.append(accuracy)

results = pd.DataFrame({
    "k": k_values,
    "accuracy": accuracies
})

results

,k,accuracy
0,1,0.402367
1,2,0.437870
2,3,0.378698
3,4,0.355030
4,5,0.313609
5,6,0.325444
6,7,0.313609
7,8,0.313609
8,9,0.349112
9,10,0.337278


In [22]:
best_row = results.loc[results["accuracy"].idxmax()]

best_k = int(best_row["k"])
best_accuracy = best_row["accuracy"]

print("Best k:", best_k)
print("Best accuracy:", best_accuracy)

Best k: 2
Best accuracy: 0.4378698224852071


### Q2.3 Selecting k
I tested values of k from 1 through 30 and compared their classification accuracy on the test set. The value with the highest accuracy was selected as the optimal k.

In [23]:
best_model = KNeighborsClassifier(n_neighbors=best_k)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

In [24]:
from sklearn.metrics import confusion_matrix, accuracy_score

confusion = pd.crosstab(
    y_test,
    y_pred,
    rownames=["Actual"],
    colnames=["Predicted"]
)

print(confusion)

accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", accuracy)

Predicted   1   2   3  4  5
Actual                     
1          25   0   6  4  1
2           0  32   0  3  0
3          12   2   9  6  4
4          12   5   8  8  0
5          15   2  10  5  0

Accuracy: 0.4378698224852071


In [25]:
correct_by_class = confusion.values.diagonal()
total_by_class = confusion.sum(axis=1).values

class_results = pd.DataFrame({
    "mine_type": confusion.index,
    "correct": correct_by_class,
    "total": total_by_class,
    "accuracy": correct_by_class / total_by_class
})

class_results

,mine_type,correct,total,accuracy
0,1,25,36,0.694444
1,2,32,35,0.914286
2,3,9,33,0.272727
3,4,8,33,0.242424
4,5,0,32,0.000000


### Q2.4 Model performance
The optimal k-NN model has an overall accuracy of about 43.8%. The confusion table shows that performance varies by mine type, and some mine types are confused with other classes more frequently than others.

This means overall accuracy alone does not completely describe the model's performance because the model may perform relatively well for one class while performing poorly for another.

### Q2.5 Using the model in practice
I would not use this model by itself to make a final decision about the type of a land mine because incorrect classifications could have serious safety consequences. Instead, I would use the prediction as supporting information along with additional measurements or expert inspection, especially for mine types that the confusion table shows are frequently misclassified.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** 1fcd0cf722a9a8704764a9592597522a7a1322c7
- **AI tool and model used:** GPT 5.6 Sol

### *Prompts used

1. Review my Phase 1 answers for Q1 and Q2 of this k-nearest neighbors assignment. Identify the most important conceptual or methodological improvements I could make, especially regarding model evaluation, selecting k, interpreting the confusion matrix, and using the model in practice. Do not change my Phase 1 work.

2. Based on the rubric, give me specific revisions that address the main improvements, any limitations in the AI suggestions, how I should independently verify the changes, and any remaining concerns or edge cases. Keep the solution appropriate for the concepts covered in this assignment.

### *AI-proposed changes


1. Revise Q1.5 to distinguish between using validation data to select model settings such as k and using a final test set to estimate performance on unseen data. Repeatedly selecting k based on the final test set can make the reported test performance overly optimistic.

2. Improve Q2's selection of k by using cross-validation on the training data rather than selecting k directly from the held-out test data. After selecting k, evaluate the chosen model once on the test set.

3. Expand the confusion-matrix interpretation by reporting class-specific performance. The original model performs much better for mine types 1 and 2 than for types 3, 4, and especially 5.

4. Strengthen the practical-use discussion by explaining that the low overall accuracy and severe class-specific errors make the classifier unsuitable as an independent safety-critical decision system.

5. Keep the original 50/50 train-test split and stratification because they follow the assignment instructions and help maintain representation of all five mine classes.

6. Note that k-NN depends on feature distances. Because voltage, height, and soil are measured on similar normalized ranges in this dataset, feature scale is less problematic here, but scaling should still be considered when applying k-NN to differently scaled features.

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| Use validation/cross-validation rather than the final test set to select k | Accept | This avoids repeatedly using the final test set while tuning the model. I verified that model selection should occur using training/validation data, while the held-out test set should be used for final evaluation. |
| Use cross-validation on the training set to select k | Accept | This gives a more reliable estimate than choosing k from one test-set result. I will compare cross-validation accuracy for candidate k values and then evaluate the selected model on the unchanged test set. |
| Add class-specific confusion-matrix interpretation | Accept | I verified this directly from my confusion table. Mine type 1 had 25/36 correct, type 2 had 32/35 correct, type 3 had 9/33 correct, type 4 had 8/33 correct, and type 5 had 0/32 correct. |
| Strengthen the practical-use warning | Accept | The confusion matrix and overall accuracy support this suggestion. A model with about 43.8% accuracy and zero correct predictions for one class should not independently determine a safety-critical action. |
| Keep the 50/50 stratified split | Accept | This follows the assignment instructions. I also checked that the resulting training and testing sets each contain 169 observations and stratification preserves representation of the five classes. |
| Standardize all features | Modify | Scaling is important for k-NN, but the EDA shows voltage, height, and soil are already roughly on a 0-to-1 scale. Therefore I would not claim that scaling is the main cause of the model's poor performance, although it remains an important consideration for other datasets. |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

Q1.1: Regression predicts a continuous numerical value, while classification predicts a categorical class label. For example, predicting a mine's numerical measurement would be regression, while predicting mine type 1 through 5 is classification.

Q1.2: A confusion table compares the actual class labels with the model's predicted class labels. It shows not only how many predictions are correct, but also which classes are being confused with one another, so it gives more detail than overall accuracy alone.

Q1.3: SSE, or Sum of Squared Errors, measures the total squared difference between predicted and actual numerical values. A smaller SSE indicates that the model's predictions are generally closer to the observed values, although SSE should be evaluated on unseen data when comparing models.

Q1.4: Overfitting occurs when a model fits the training data too closely and does not generalize well to new data. Underfitting occurs when a model is too simple to capture important relationships in the data, causing poor performance even on the training data.

Q1.5: Splitting the data allows us to evaluate whether a model generalizes to observations it did not train on. Model choices such as k should ideally be made using validation data or cross-validation within the training data, while a separate test set should be saved for final evaluation; repeatedly choosing the model based on the test set can make the reported test performance overly optimistic.

Q1.6: A predicted class label is simple and easy to use because it provides one clear classification, but it hides uncertainty about the prediction. A probability distribution provides more information by showing how strongly the model supports each possible class, which is useful when predictions are uncertain, although it is more complicated to interpret and the estimated probabilities may not always be well calibrated.

In [26]:
#Q2.1
import pandas as pd

df = pd.read_csv("land_mines.csv")
df.head()

,voltage,height,soil,mine_type
0,0.338157,0.000000,0.0,1
1,0.320241,0.181818,0.0,1
2,0.287009,0.272727,0.0,1
3,0.256284,0.454545,0.0,1
4,0.262840,0.545455,0.0,1


### Q2.1 Load the data and perform EDA
The dataset contains three predictor variables, `voltage`, `height`, and `soil`, and the target variable `mine_type`, which contains classes 1 through 5.

In [27]:
print(df.shape)
print(df.info())
print(df.describe())
print(df["mine_type"].value_counts().sort_index())

(338, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
None
          voltage      height        soil   mine_type
count  338.000000  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550    2.952663
std      0.195819    0.306043    0.344244    1.419703
min      0.197734    0.000000    0.000000    1.000000
25%      0.309737    0.272727    0.200000    2.000000
50%      0.359516    0.545455    0.600000    3.000000
75%      0.482628    0.727273    0.800000    4.000000
max      0.999999    1.000000    1.000000    5.000000
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64


In [28]:
from sklearn.model_selection import train_test_split

X = df[["voltage", "height", "soil"]]
y = df["mine_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.50,
    random_state=42,
    stratify=y
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

Training observations: 169
Testing observations: 169


### Q2.2 Train/test split
I split the data 50/50 into training and testing sets. I used stratification so that each mine type is represented proportionally in both sets.

In [29]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import pandas as pd

k_values = range(1, 31)
cv_accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    cv_accuracies.append(scores.mean())

cv_results = pd.DataFrame({
    "k": k_values,
    "cv_accuracy": cv_accuracies
})

cv_results

,k,cv_accuracy
0,1,0.325134
1,2,0.390374
2,3,0.325312
3,4,0.330660
4,5,0.378253
5,6,0.384314
6,7,0.372549
7,8,0.348841
8,9,0.372549
9,10,0.384670


In [30]:
best_row = cv_results.loc[cv_results["cv_accuracy"].idxmax()]
best_k = int(best_row["k"])
best_cv_accuracy = best_row["cv_accuracy"]

print("Best k:", best_k)
print("Best cross-validation accuracy:", best_cv_accuracy)

Best k: 2
Best cross-validation accuracy: 0.3903743315508021


### Q2.3 Selecting k
Rather than selecting k directly from the final test-set accuracy, I used 5-fold cross-validation on the training data for k values from 1 through 30. I selected the k with the highest mean cross-validation accuracy. This keeps the held-out test data separate from the model-selection process and gives a better estimate of which k generalizes within the training data.

In [31]:
from sklearn.metrics import accuracy_score

best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

confusion = pd.crosstab(
    y_test,
    y_pred,
    rownames=["Actual"],
    colnames=["Predicted"]
)

print(confusion)

accuracy = accuracy_score(y_test, y_pred)
print("\nTest accuracy:", accuracy)

Predicted   1   2   3  4  5
Actual                     
1          25   0   6  4  1
2           0  32   0  3  0
3          12   2   9  6  4
4          12   5   8  8  0
5          15   2  10  5  0

Test accuracy: 0.4378698224852071


In [32]:
correct_by_class = confusion.values.diagonal()
total_by_class = confusion.sum(axis=1).values

class_results = pd.DataFrame({
    "mine_type": confusion.index,
    "correct": correct_by_class,
    "total": total_by_class,
    "accuracy": correct_by_class / total_by_class
})

class_results

,mine_type,correct,total,accuracy
0,1,25,36,0.694444
1,2,32,35,0.914286
2,3,9,33,0.272727
3,4,8,33,0.242424
4,5,0,32,0.000000


### Q2.4 Model performance
After selecting k using cross-validation on the training data, I fit the final k-NN model and evaluated it once on the held-out test set. The final test accuracy was **43.8%**.

The confusion table shows that performance is not equal across mine types. Mine type **2** had the strongest performance, while mine type **5** had the weakest performance. This shows why overall accuracy alone is not enough to evaluate this classifier: a model can make many correct predictions for some classes while still making serious errors for others.

### Q2.5 Using the model in practice
I would not use this k-NN model by itself to determine the type of a land mine because a classification error in this setting could have serious safety consequences. The confusion matrix should be used to identify the types for which the model is especially unreliable rather than relying only on overall accuracy.

In practice, I would treat the model's prediction as supporting information and combine it with additional sensor measurements, expert inspection, or another validated detection method before making a decision. Cases involving mine types with poor class-specific performance should receive especially careful review. The model may therefore be useful as one component of a larger decision process, but the results do not support using it as an autonomous classification system.

### *Reflection on AI assistance
In your own words without generative AI.

The main change that was suggested by AI was changing how exactly I selected k. In Phase 1, I tested multiple k values on the test set and chose the one with the highest accuracy. Also ChatGPT said that this uses information from the test set during model selection repeatedly. I changed my approach so that I select k using cross validation on the training data and the test data is only used for the final evaluation.



ChatGPT also said that I should look more closesly at class specific performance instead of using only overall accuracy. I checked this using my confusion matrix and the accuracy calculated separately for each mine type. This proved that the model's performancy varies a lot between classes, which can make the consuion matrix important.



A limitation I found with AI was its unnecessary emphasis on featury scaling. I checked the statistics and found that voltage, height, and soil already have similar ranges, so I did not treat it as the main issue with the dataset.



I guess a final concern is that the model may not classify every mine type reliably. Because of this being an important safety decision, I would not use the model as the only source for making an actual decision.